# Periodic Darcy: separate face, local and reference refinement

This case follows Paredes, Valentin and Versieux (2017), section 4, DOI
[10.1090/mcom/3108](https://doi.org/10.1090/mcom/3108). On the unit square,
$K=1+100\cos^2(150x)\sin^2(150y)$, $f=\sin(x)\sin(y)$ and boundary pressure is zero.
The fixed MHM grid has 8×8 squares, local continuous Q1 fields, and 1–32 constant
normal-flux modes per face. The historical local refinement is unspecified.

The classical reference has its own refinement study. All comparisons below
are integrated full broken H1 or L2 norms, not norms of plotted sample arrays.


The main equations execute from source and state their current discretization. The original archived comparisons below execute with `--historical` and require their exact attributed payloads. `--study` runs the complete public current acquisition declared in the catalogue, with its own output provenance.


In [ ]:
from pathlib import Path
import sys

# Locate the checkout from the repository root or any notebook subdirectory.
ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").is_file()
     and (path / "src" / "pymhm").is_dir()
     and (path / "notebooks").is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError("Run this notebook from the PyMHM checkout.")
root = ROOT
for directory in (ROOT, ROOT / "examples"):
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))

import os
HISTORICAL_REPLAY = os.environ.get("PYMHM_NOTEBOOK_HISTORICAL", "0") == "1"
STUDY_REPLAY = os.environ.get("PYMHM_NOTEBOOK_STUDY", "0") == "1"


## Current periodic problem through the public equations

The following acquisition uses the stated oscillatory permeability, source and
zero exterior pressure on the complete unit square. The finite factor sum is
exact; it performs no material averaging or rank approximation. Continuous Q1
reference equations are assembled independently of the MHM trace restriction.
Their three mesh levels report their own full L2 and H1 increments. These quick
levels are sensitivity controls: a small linear residual alone does not make a
fine numerical field an accurate continuum reference. The full refinement
recipe under `--study` retains the larger reference and MHM acquisitions.

$$
\begin{aligned}
a(u,v)&=\int_\Omega K\nabla u\cdot\nabla v,
&L(v)&=\int_\Omega \sin(x)\sin(y)v,\\
K(x,y)&=1+100\cos^2(150x)\sin^2(150y),
&u\vert_{\partial\Omega}&=0.
\end{aligned}
$$


In [ ]:
import numpy as np
from threadpoolctl import threadpool_limits
from pymhm.materials.separable import SeparableField
from pymhm.meshes.cartesian import CartesianMacroMesh
from examples.formulations.conforming import conforming_separated
from examples.periodic_norms import difference
from examples.verify_periodic import factor_x, factor_y

permeability = SeparableField(((1.0, 1.0), (factor_x, factor_y)))
forcing = SeparableField(((np.sin, np.sin),))
with threadpool_limits(1):
    current_references = [
        conforming_separated(
            CartesianMacroMesh(n), degree=1, permeability=permeability,
            source=forcing, dirichlet=0.0, quadrature_order=12,
        )
        for n in (64, 128, 256)
    ]
    increments = [
        difference(fine, (coarse,))
        for coarse, fine in zip(current_references[:-1], current_references[1:], strict=True)
    ]
assert max(field.residual for field in current_references) < 1e-10
print("Independent Q1 reference increments (fine norm denominators):", increments)
print("These coarse increments must be checked before using a numerical baseline.")


In [ ]:
from pathlib import Path
import json
import numpy as np
from IPython.display import Image, display
from pymhm.meshes.cartesian import CartesianMacroMesh


## Small independent affine patch

An affine pressure is exactly represented by Q1 locals and constant flux traces.
This portable patch runs here; the larger periodic fields are replayed from their
numerical records, without requiring a high-memory solve in the notebook.


## Declare local and global forms

The application defines the energy/source and both oriented trace incidences.
Qk Cartesian and Pk tetrahedral coordinates use their shared FEM kernels:

$$
\begin{aligned}
a_T(p,v)&=(K\nabla p,\nabla v)_T, & L_T(v)&=(f,v)_T,\\
b_T(\lambda,v)&=\langle\lambda,v\rangle_{\partial T}, &
c_T(p,\mu)&=-\langle p,\mu\rangle_{\partial T}.
\end{aligned}
$$

The declared `LocalEquations` use C=-B.T, a constant pressure kernel and its
physical integral moment. The additional global `Equation` supplies the weak
Dirichlet pressure functional with its prescribed sign. Neumann data are
outward physical fluxes; a fully Neumann boundary uses an explicit pressure
integral constraint. Generic assembly and field interpretation are separate
calls. The sampled gradient flux remains distinct from the conservative trace.


In [ ]:
from pymhm import assemble
from examples.formulations.darcy import pressure_constraints
from examples.formulations.cartesian_darcy import define_cartesian_darcy, recover_cartesian_darcy

exact = lambda points: 1 + points[:, 0] - 2 * points[:, 1]
patch_definition = define_cartesian_darcy(
    CartesianMacroMesh(2), local_refinement=4, dirichlet=exact
)
patch_system = assemble(patch_definition.problem)
patch = recover_cartesian_darcy(
    patch_definition,
    patch_system,
    patch_system.solve(constraints=pressure_constraints(patch_definition, patch_system)),
)
assert patch.l2_error(exact) < 1e-11
assert patch.flux_l2_error([-1, 2]) < 1e-10
print({"pressure_L2_error": patch.l2_error(exact), "residual": patch.hybrid.residual})


## Executed original-equation and replay controls

The current numerical records contain Q1/P0 fields at r128 and r256, with all
six published segmented trace spaces. Their full original volume and weak-trace
rows satisfy the declared physical-source residual criterion. The independent
full Q1/P0 assemblies agree on the same spaces. This establishes algebraic
agreement, while the r128-to-r256 full H1 increment remains about 19%.

The affine patch above is a separate invariant, not a replacement for this
heterogeneous case or a reproduction of its published curve.


In [ ]:
if HISTORICAL_REPLAY:
    controls = json.loads((root / "examples/results/periodic/streamed-original-verification.json").read_text())
    assert controls["accepted"]
    assert {case["refinement"] for case in controls["cases"]} == {128, 256}
    for case in controls["cases"]:
        rows = case["original_equation_controls"]
        assert {row["segments"] for row in rows} == {1, 2, 4, 8, 16, 32}
        for row in rows:
            original = row["final_original_equations"]
            assert original["accepted"]
            assert original["original_saddle_relative_residual"] <= controls["original_physical_source_relative_criterion"]
            replay = row["numeric_replay_digests"]
            assert replay["field_trace_coarse_kernel_digests_before"] == replay["field_trace_coarse_kernel_digests_after"]
            print(f"r={case['refinement']:4d} s={row['segments']:2d} "
                  f"original relative residual={original['original_saddle_relative_residual']:.8g}")
    for row in controls["local_refinement_increment"]:
        print(f"s={row['segments']:2d} r128-to-r256 "
              f"full broken H1 increment={row['relative_full_broken_h1']:.8g}")

    reference = json.loads((root / "examples/results/periodic/q1-reference-verification.json").read_text())
    assert reference["historical_reference_space_executed"]
    assert not reference["continuum_reference_resolution_demonstrated"]
    assert not reference["literature_reproduction_accepted"]
    for row in reference["acquisitions"]:
        assert row["original_physical_free_row_relative_residual"] <= reference["original_criterion"]
        print(f"Conforming Q1/{row['subdivisions']} "
              f"original relative residual={row['original_physical_free_row_relative_residual']:.8g}")
    for row in reference["own_spatial_refinement"]:
        print(f"Q1/{row['coarse']} to Q1/{row['fine']}: "
              f"relative full H1 increment={row['relative_h1']:.8g}, "
              f"relative L2 increment={row['relative_l2']:.8g}")


## Published curve and selected physical panels

The publication markers come from vector-PDF coordinates with an extraction
allowance. The four printed annotations occur at s1, s4, s16 and s32; s2 and
s8 retain digitization uncertainty. These allowances are distinct from
numerical reference error.

Q1/4096 executes the historical conforming reference resolution, but its
Q1/2048-to-Q1/4096 full H1 increment remains 9.77%. It is not a certified exact
or resolved baseline. The historical local subdivision count is unavailable;
matched literature reproduction and final norm-comparison panels remain pending
adequate local and reference resolution.

The selected panels retain actual macro boundaries, independent one-sided
finite-element interface values and separate signed physical Darcy-flux scales.
The raw constitutive flux is not an H(div) reconstruction.


In [ ]:
if HISTORICAL_REPLAY:
    for name in ("problem", "convergence", "fields"):
        display(Image(filename=str(root / f"docs/figures/periodic/{name}.png")))


## Reproduce the larger campaign

Use `examples/periodic_reference.py --degree 1 --order 4` for guarded conforming
Q1 acquisitions; its default remains Q5/order10. The archives retain executed
basis matrices, their digests and effective coefficient/operator precision.
`examples/verify_periodic.py` owns phased MHM acquisitions,
`examples/compare_periodic.py` computes exact common-partition polynomial norms,
and `examples/plot_periodic.py` renders figures using the validated readers.

The periodic case page gives commands, quadrature and measured memory use.
Large fields remain under `build/results`; compact controls carry their actual
source generations and basis/replay checks. New acquisitions do not retag
older coefficients. Reference and local increments are sensitivities rather
than certified bounds on the unknown continuum error.
